# A Comparison of Modern Explainable AI (XAI) Frameworks

This notebook provides a hands-on comparison of five popular XAI frameworks: LIME, SHAP, ELI5, InterpretML, and Captum. We will use a synthetic dataset to train both a tree-based model (for the model-agnostic frameworks) and a neural network (for Captum) and then apply each framework to explain the models' behavior.

## 1. Setup and Data Generation

First, let's install the necessary libraries and generate a synthetic dataset for a regression problem.

In [ ]:
%pip install lime shap eli5 interpret torch captum scikit-learn numpy pandas

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
import torch
import torch.nn as nn
import lime
import lime.lime_tabular
import shap
import eli5
from interpret.ext.blackbox import TabularExplainer
from captum.attr import IntegratedGradients

# For consistent results
np.random.seed(42)

In [ ]:
# Generate a synthetic dataset
X, y = make_regression(n_samples=200, n_features=10, n_informative=5, random_state=42)
feature_names = [f'feature_{i}' for i in range(X.shape[1])]
X_df = pd.DataFrame(X, columns=feature_names)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42)

## 2. Model Training

We will train two models:
1. A `RandomForestRegressor` for the model-agnostic frameworks.
2. A simple PyTorch neural network for Captum.

In [ ]:
# Train the Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
# Define and train the PyTorch model
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(10, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

torch_model = SimpleNet()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(torch_model.parameters(), lr=0.01)

for epoch in range(100):
    optimizer.zero_grad()
    outputs = torch_model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

## 3. LIME (Local Interpretable Model-agnostic Explanations)

LIME explains a single prediction by creating a simpler, interpretable model (e.g., a linear model) in the local vicinity of the instance being explained. It perturbs the input data and sees how the model's predictions change to infer feature importance.

In [ ]:
# Create a LIME explainer
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=feature_names,
    class_names=['prediction'],
    mode='regression'
)

# Explain a single instance from the test set
instance_to_explain = X_test.iloc[0]
lime_explanation = lime_explainer.explain_instance(
    instance_to_explain.values,
    rf_model.predict,
    num_features=5
)

# Show the explanation
lime_explanation.show_in_notebook(show_table=True)

## 4. SHAP (SHapley Additive exPlanations)

SHAP (SHapley Additive exPlanations) is a game-theoretic approach to explain the output of any machine learning model. It connects optimal credit allocation with local explanations using the classic Shapley values and their related extensions.

In [ ]:
# Create a SHAP explainer for the Random Forest model
# For tree-based models, TreeExplainer is much faster than the generic KernelExplainer
shap_explainer = shap.TreeExplainer(rf_model)
shap_values = shap_explainer.shap_values(X_test)

# Initialize javascript for shap plots in the notebook
shap.initjs()

### Global Explanation: Summary Plot

The summary plot shows the most important features for the model globally. Each point represents a single feature's SHAP value for a specific instance.

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)

### Local Explanation: Force Plot

The force plot shows how each feature contributed to the prediction for a single instance. Features pushing the prediction higher are in red, and those pushing it lower are in blue.

In [ ]:
shap.force_plot(shap_explainer.expected_value, shap_values[0,:], X_test.iloc[0,:])

## 5. ELI5 (Explain Like I'm 5)

ELI5 is a library that helps to debug machine learning classifiers and explain their predictions. It provides support for scikit-learn models and can show global feature importances as well as explain individual predictions.

### Global Explanation: Feature Importances

In [ ]:
eli5.show_weights(rf_model, feature_names=feature_names)

### Local Explanation: Prediction Breakdown

In [ ]:
eli5.show_prediction(rf_model, X_test.iloc[0], feature_names=feature_names)

## 6. InterpretML

InterpretML is an open-source package that provides a comprehensive suite of tools for model interpretability. It offers both 'glassbox' models that are inherently interpretable and methods for explaining 'blackbox' models. Its interactive dashboard is one of its key features.

In [ ]:
from interpret import show

# Create a TabularExplainer for the Random Forest model
tabular_explainer = TabularExplainer(rf_model, X_train, features=feature_names)

# Get a global explanation
global_explanation = tabular_explainer.explain_global(X_test)

# Show the global explanation dashboard
show(global_explanation)

## 7. Captum

Captum is a model interpretability and understanding library for PyTorch. It provides a suite of powerful algorithms, including Integrated Gradients, to explain the behavior of neural networks.

### Feature Attribution with Integrated Gradients

In [ ]:
# Create an Integrated Gradients explainer
ig = IntegratedGradients(torch_model)

# Choose an instance to explain and compute attributions
instance_to_explain_tensor = X_test_tensor[0].unsqueeze(0)
attributions, delta = ig.attribute(instance_to_explain_tensor, return_convergence_delta=True)
attributions = attributions.detach().numpy().flatten()

# Visualize the attributions
pd.DataFrame({'Feature': feature_names, 'Attribution': attributions}).set_index('Feature').plot(kind='barh', title='Captum Integrated Gradients Attributions');

## 8. Comparison Table

| Framework   | Pros                                                                                                 | Cons                                                                                      | Best Use-Case                                                                                             |
|-------------|------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------|
| **LIME**    | - Very intuitive and easy to understand.<br>- Truly model-agnostic.<br>- Good for explaining individual predictions. | - Can be slow.<br>- Explanations can be unstable.<br>- Primarily for local explanations.          | Explaining a single prediction to a non-technical audience; building trust in individual model outputs. |
| **SHAP**    | - Strong theoretical foundation (Shapley values).<br>- Provides both local and global explanations.<br>- Consistent and accurate. | - Can be computationally expensive, especially with non-tree models.<br>- Can be complex to interpret. | When you need robust, consistent explanations for both individual predictions and overall model behavior. |
| **ELI5**    | - Very easy to use.<br>- Great for scikit-learn models.<br>- Provides clear feature importances.         | - Limited support for models outside of the scikit-learn ecosystem.<br>- Less visual than others. | Quickly debugging and understanding the feature importances of scikit-learn models.                     |
| **InterpretML** | - Comprehensive suite of tools.<br>- Interactive dashboard for exploration.<br>- Includes interpretable "glassbox" models. | - Can be more complex to set up.<br>- Dashboard can be overwhelming.                       | When you need a full suite of interpretability tools, including an interactive dashboard for deep dives.  |
| **Captum**  | - Built specifically for PyTorch.<br>- Wide range of attribution algorithms.<br>- Very flexible and powerful. | - Steeper learning curve.<br>- Primarily for deep learning models.<br>- Requires more code to use. | Explaining the behavior of complex PyTorch models, especially in research and development settings.      |